# 🚀 GNN Model Training for Academic Research Assistant (Aura)

## **Kaggle GPU-Optimized Notebook**

This notebook trains a Graph Neural Network on academic research data (Papers, Authors, Concepts) for:
1. **Semantic Plagiarism Detection**
2. **Research Discovery Engine**

---

### 📋 **Before You Start:**

1. **Enable GPU Acceleration:**
   - Go to Kaggle Notebook Settings → Accelerator → Select **GPU T4 x2** (Free!)
   
2. **Upload Your Data Files:**
   - Click "Add Data" → "Upload" → Upload `nodes.csv` and `relationships.csv`
   - Or use Kaggle Datasets if you've created a dataset

3. **Expected Training Time:** 5-10 minutes with GPU 🔥

---

### 📊 **Model Architecture:**
- **GNN Type:** GraphSAGE (2-layer)
- **Hidden Channels:** 256
- **Output Channels:** 128
- **Task:** Link Prediction (Paper → Concept)
- **Training:** Binary Cross-Entropy with negative sampling

Let's get started! 👇

## 1️⃣ Check GPU Availability

In [ ]:
import torch
import sys

print("=" * 80)
print("SYSTEM INFORMATION")
print("=" * 80)

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = torch.device('cuda')
    print("\n✅ GPU ACCELERATION ENABLED! Training will be FAST! 🚀")
else:
    device = torch.device('cpu')
    print("\n⚠️  GPU not detected. Training will be slower on CPU.")
    print("💡 Tip: Enable GPU in Settings → Accelerator → GPU T4 x2")

print(f"\nPython Version: {sys.version}")
print("=" * 80)

## 2️⃣ Install Required Libraries

Installing PyTorch Geometric and other dependencies for GNN training.

In [ ]:
# Install PyTorch Geometric and dependencies
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install -q torch-geometric
!pip install -q sentence-transformers

print("\n✅ All libraries installed successfully!")

## 3️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import os
from pathlib import Path

print("✅ All imports successful!")

## 4️⃣ Load Data Files

**📌 IMPORTANT:** Make sure you've uploaded `nodes.csv` and `relationships.csv` to your Kaggle notebook!

**How to upload:**
1. Click "Add Data" button (top right)
2. Click "Upload" → Select your CSV files
3. Or create a Kaggle Dataset and add it to this notebook

The files should be in `/kaggle/input/` directory.

In [ ]:
# Try to find data files in common Kaggle locations
data_paths = [
    '/kaggle/input/',  # Kaggle dataset location
    '/kaggle/working/',  # Kaggle working directory
    './'  # Current directory (if uploaded directly)
]

nodes_file = None
rels_file = None

for path in data_paths:
    if os.path.exists(path):
        # List all subdirectories
        for root, dirs, files in os.walk(path):
            if 'nodes.csv' in files and nodes_file is None:
                nodes_file = os.path.join(root, 'nodes.csv')
            if 'relationships.csv' in files and rels_file is None:
                rels_file = os.path.join(root, 'relationships.csv')

if nodes_file and rels_file:
    print(f"✅ Found data files!")
    print(f"   Nodes: {nodes_file}")
    print(f"   Relationships: {rels_file}")
else:
    print("❌ Data files not found!")
    print("\n📋 Please upload your data files:")
    print("   1. Click 'Add Data' button")
    print("   2. Upload nodes.csv and relationships.csv")
    print("   3. Re-run this cell")
    
# Load data
print("\n" + "="*80)
print("Loading data from CSV files...")
print("="*80)

nodes_df = pd.read_csv(nodes_file)
rels_df = pd.read_csv(rels_file)
nodes_df['text'] = nodes_df['text'].fillna('')

print(f"\n✅ Data loaded successfully!")
print(f"   Total Nodes: {len(nodes_df)}")
print(f"   Total Relationships: {len(rels_df)}")
print(f"\n📊 Node Distribution:")
for label, count in nodes_df['label'].value_counts().items():
    print(f"   - {label}: {count}")
print(f"\n🔗 Relationship Distribution:")
for rel_type, count in rels_df['type'].value_counts().items():
    print(f"   - {rel_type}: {count}")

## 5️⃣ Create Text Embeddings

Using SentenceTransformer to create 384-dimensional embeddings for all paper texts.
This will take a few minutes depending on the number of papers.

In [ ]:
print("="*80)
print("Creating text embeddings for papers...")
print("="*80)

# Load pre-trained SentenceTransformer model
text_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ Loaded SentenceTransformer model: all-MiniLM-L6-v2")
print(f"   Embedding dimension: 384")

# Get paper texts
paper_df = nodes_df[nodes_df['label'] == 'Paper']
paper_texts = paper_df['text'].tolist()
num_papers = len(paper_texts)

print(f"\n📝 Encoding {num_papers} papers...")
print("   This may take 2-5 minutes...")

# Create embeddings (with progress bar)
paper_embeddings = text_model.encode(
    paper_texts, 
    show_progress_bar=True,
    batch_size=32,
    device=str(device)  # Use GPU if available
)

print(f"\n✅ Embeddings created!")
print(f"   Shape: {paper_embeddings.shape}")
print(f"   Memory: {paper_embeddings.nbytes / 1e6:.2f} MB")

## 6️⃣ Build Heterogeneous Graph

Creating a heterogeneous graph with:
- **Paper** nodes (with text embeddings)
- **Author** nodes (one-hot encoded)
- **Concept** nodes (one-hot encoded)
- **WROTE** edges (Author → Paper)
- **HAS_CONCEPT** edges (Paper → Concept)

In [ ]:
print("="*80)
print("Building heterogeneous graph...")
print("="*80)

# Create HeteroData object
data = HeteroData()
node_mappings = {}

# Create node features for each node type
for label in nodes_df['label'].unique():
    mask = nodes_df['label'] == label
    df_type = nodes_df[mask]
    
    # Create mapping from original node IDs to sequential indices
    node_mappings[label] = {node_id: i for i, node_id in enumerate(df_type['nodeId'])}
    
    if label == 'Paper':
        # Use pre-computed embeddings for papers
        data[label].x = torch.tensor(paper_embeddings, dtype=torch.float)
        print(f"✅ Paper nodes: {len(df_type)} (text embeddings: {data[label].x.shape})")
    else:
        # One-hot encoding for Author and Concept nodes
        data[label].x = torch.eye(len(df_type), dtype=torch.float)
        print(f"✅ {label} nodes: {len(df_type)} (one-hot encoded: {data[label].x.shape})")

# Create edges with BIDIRECTIONAL relationships
schema = {
    'WROTE': ('Author', 'wrote', 'Paper'),
    'WRITTEN_BY': ('Paper', 'written_by', 'Author'),  # Reverse edge!
    'HAS_CONCEPT': ('Paper', 'has_concept', 'Concept'),
    'CONCEPT_OF': ('Concept', 'concept_of', 'Paper')  # Reverse edge!
}

print("\n🔗 Creating edges (with bidirectional relationships)...")
for edge_type_str, (source_label, rel_name, target_label) in schema.items():
    # Check if this is a reverse edge
    if edge_type_str == 'WRITTEN_BY':
        # Use WROTE edges but reverse source/target
        mask = rels_df['type'] == 'WROTE'
        if mask.any():
            source_indices = rels_df[mask]['target'].map(node_mappings[source_label]).values  # Reversed!
            target_indices = rels_df[mask]['source'].map(node_mappings[target_label]).values  # Reversed!
        else:
            continue
    elif edge_type_str == 'CONCEPT_OF':
        # Use HAS_CONCEPT edges but reverse source/target
        mask = rels_df['type'] == 'HAS_CONCEPT'
        if mask.any():
            source_indices = rels_df[mask]['target'].map(node_mappings[source_label]).values  # Reversed!
            target_indices = rels_df[mask]['source'].map(node_mappings[target_label]).values  # Reversed!
        else:
            continue
    else:
        # Regular edges
        mask = rels_df['type'] == edge_type_str
        if not mask.any():
            print(f"   ⚠️  Warning: No {edge_type_str} edges found")
            continue
        source_indices = rels_df[mask]['source'].map(node_mappings[source_label]).values
        target_indices = rels_df[mask]['target'].map(node_mappings[target_label]).values
    
    # Remove NaN values (unmapped nodes)
    valid_mask = ~(pd.isna(source_indices) | pd.isna(target_indices))
    source_indices = source_indices[valid_mask].astype(int)
    target_indices = target_indices[valid_mask].astype(int)
    
    # Create edge index tensor
    edge_index = torch.tensor([source_indices, target_indices], dtype=torch.long)
    data[source_label, rel_name, target_label].edge_index = edge_index
    
    arrow = "⟷" if edge_type_str in ['WRITTEN_BY', 'CONCEPT_OF'] else "→"
    print(f"   ✅ {edge_type_str}: {edge_index.shape[1]:,} edges ({source_label} {arrow} {target_label})")

print(f"\n📊 Graph Summary:")
print(f"   Node types: {data.node_types}")
print(f"   Edge types: {data.edge_types}")
print(f"   Metadata: {data.metadata()}")
print("\n💡 All node types now participate in message passing!")

## 7️⃣ Prepare Training/Validation Data

Splitting Paper→Concept edges for link prediction task with negative sampling.

In [ ]:
print("="*80)
print("Preparing training/validation splits...")
print("="*80)

# Get paper-concept edges for training
paper_concept_edges = data['Paper', 'has_concept', 'Concept'].edge_index.t().numpy()
print(f"Total Paper→Concept edges: {len(paper_concept_edges):,}")

# Split into train/val (80/20)
train_edges, val_edges = train_test_split(
    paper_concept_edges, 
    test_size=0.2, 
    random_state=42
)

print(f"\n📊 Split:")
print(f"   Training edges: {len(train_edges):,} (80%)")
print(f"   Validation edges: {len(val_edges):,} (20%)")

# Create negative samples
num_papers = data['Paper'].x.shape[0]
num_concepts = data['Concept'].x.shape[0]

def create_negative_samples(num_samples, existing_edges):
    """Create random negative samples that don't exist in the graph"""
    existing_set = set(map(tuple, existing_edges))
    negatives = []
    
    while len(negatives) < num_samples:
        paper_idx = np.random.randint(0, num_papers)
        concept_idx = np.random.randint(0, num_concepts)
        if (paper_idx, concept_idx) not in existing_set:
            negatives.append([paper_idx, concept_idx])
    
    return np.array(negatives)

print("\n🎲 Creating negative samples...")
train_neg = create_negative_samples(len(train_edges), paper_concept_edges)
val_neg = create_negative_samples(len(val_edges), paper_concept_edges)

# Convert to tensors
train_pos_edge_index = torch.tensor(train_edges.T, dtype=torch.long)
train_neg_edge_index = torch.tensor(train_neg.T, dtype=torch.long)
val_pos_edge_index = torch.tensor(val_edges.T, dtype=torch.long)
val_neg_edge_index = torch.tensor(val_neg.T, dtype=torch.long)

print(f"   Training: {len(train_edges):,} positive + {len(train_neg):,} negative")
print(f"   Validation: {len(val_edges):,} positive + {len(val_neg):,} negative")
print(f"\n✅ Data preparation complete!")

## 8️⃣ Define GNN Model Architecture

Building a 2-layer GraphSAGE model for heterogeneous graphs.

In [ ]:
class GNN(torch.nn.Module):
    """Graph Neural Network with 2 GraphSAGE layers"""
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
    
    def forward(self, x, edge_index):
        # First convolution + ReLU
        x = self.conv1(x, edge_index).relu()
        # Dropout for regularization
        x = F.dropout(x, p=0.5, training=self.training)
        # Second convolution
        return self.conv2(x, edge_index)

class Classifier(torch.nn.Module):
    """Edge classifier for link prediction"""
    def forward(self, x_paper, x_concept, edge_label_index):
        # Get embeddings for paper and concept nodes
        edge_feat_paper = x_paper[edge_label_index[0]]
        edge_feat_concept = x_concept[edge_label_index[1]]
        # Dot product to predict edge existence
        return (edge_feat_paper * edge_feat_concept).sum(dim=-1)

print("✅ Model architecture defined!")
print("\nGNN Architecture:")
print("  Layer 1: SAGEConv (variable input → 256 hidden)")
print("  Activation: ReLU")
print("  Dropout: 0.5")
print("  Layer 2: SAGEConv (256 → 128 output)")
print("\nClassifier:")
print("  Method: Dot product between paper and concept embeddings")

## 9️⃣ Initialize Models and Optimizer

In [ ]:
print("="*80)
print("Initializing models...")
print("="*80)

# Initialize GNN model
gnn_model = GNN(hidden_channels=256, out_channels=128)

# Convert to heterogeneous model
gnn_model = to_hetero(gnn_model, data.metadata(), aggr='sum')

# Initialize classifier
classifier = Classifier()

# Move models to GPU/CPU
gnn_model = gnn_model.to(device)
classifier = classifier.to(device)
data = data.to(device)

# Move edge indices to device
train_pos_edge_index = train_pos_edge_index.to(device)
train_neg_edge_index = train_neg_edge_index.to(device)
val_pos_edge_index = val_pos_edge_index.to(device)
val_neg_edge_index = val_neg_edge_index.to(device)

# IMPORTANT: Initialize lazy parameters by doing a forward pass
print("\n🔧 Initializing model parameters...")
with torch.no_grad():
    _ = gnn_model(data.x_dict, data.edge_index_dict)
print("   ✓ Parameters initialized!")

# Initialize optimizer (Adam with learning rate 0.001)
optimizer = torch.optim.Adam(
    list(gnn_model.parameters()) + list(classifier.parameters()), 
    lr=0.001
)

# Count parameters (now they're initialized)
total_params = sum(p.numel() for p in gnn_model.parameters())
trainable_params = sum(p.numel() for p in gnn_model.parameters() if p.requires_grad)

print(f"\n✅ Models initialized!")
print(f"   Device: {device}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Optimizer: Adam (lr=0.001)")

## 🔟 Define Training and Validation Functions

In [ ]:
def train_epoch():
    """Train for one epoch"""
    gnn_model.train()
    classifier.train()
    optimizer.zero_grad()
    
    # Forward pass through GNN
    x_dict = gnn_model(data.x_dict, data.edge_index_dict)
    
    # Predict positive edges (should be close to 1)
    pos_pred = classifier(
        x_dict['Paper'], 
        x_dict['Concept'], 
        train_pos_edge_index
    )
    
    # Predict negative edges (should be close to 0)
    neg_pred = classifier(
        x_dict['Paper'], 
        x_dict['Concept'], 
        train_neg_edge_index
    )
    
    # Binary cross-entropy loss
    pos_loss = F.binary_cross_entropy_with_logits(pos_pred, torch.ones_like(pos_pred))
    neg_loss = F.binary_cross_entropy_with_logits(neg_pred, torch.zeros_like(neg_pred))
    loss = pos_loss + neg_loss
    
    # Backpropagation
    loss.backward()
    optimizer.step()
    
    return loss.item()

@torch.no_grad()
def validate():
    """Validate on validation set"""
    gnn_model.eval()
    classifier.eval()
    
    # Forward pass
    x_dict = gnn_model(data.x_dict, data.edge_index_dict)
    
    # Predict positive and negative edges
    pos_pred = classifier(x_dict['Paper'], x_dict['Concept'], val_pos_edge_index)
    neg_pred = classifier(x_dict['Paper'], x_dict['Concept'], val_neg_edge_index)
    
    # Calculate accuracy
    pos_correct = (pos_pred > 0).sum().item()
    neg_correct = (neg_pred < 0).sum().item()
    total = len(val_pos_edge_index[0]) + len(val_neg_edge_index[0])
    accuracy = (pos_correct + neg_correct) / total
    
    # Calculate loss
    pos_loss = F.binary_cross_entropy_with_logits(pos_pred, torch.ones_like(pos_pred))
    neg_loss = F.binary_cross_entropy_with_logits(neg_pred, torch.zeros_like(neg_pred))
    val_loss = (pos_loss + neg_loss).item()
    
    return val_loss, accuracy

print("✅ Training and validation functions defined!")

## 1️⃣1️⃣ Train the Model! 🚀

This is where the magic happens! Training will take 5-10 minutes with GPU.

The model learns to predict which concepts belong to which papers using:
- **Positive samples:** Actual Paper→Concept edges
- **Negative samples:** Random pairs that don't exist
- **Loss:** Binary Cross-Entropy
- **Optimization:** Adam optimizer
- **Early stopping:** Stops if validation accuracy doesn't improve for 10 epochs

In [ ]:
import time

print("="*80)
print("🚀 STARTING TRAINING!")
print("="*80)

# Training configuration
num_epochs = 50
best_val_acc = 0
best_epoch = 0
patience = 10
patience_counter = 0

# Track metrics
train_losses = []
val_losses = []
val_accuracies = []

# Start time
start_time = time.time()

print(f"\nConfiguration:")
print(f"   Max epochs: {num_epochs}")
print(f"   Early stopping patience: {patience}")
print(f"   Device: {device}")
print(f"\n{'Epoch':<8} {'Train Loss':<12} {'Val Loss':<12} {'Val Acc':<12} {'Status'}")
print("-" * 80)

for epoch in range(1, num_epochs + 1):
    # Train
    train_loss = train_epoch()
    train_losses.append(train_loss)
    
    # Validate
    val_loss, val_acc = validate()
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    # Print progress
    status = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save(gnn_model.state_dict(), 'classifier.pth')
        status = "✓ Best!"
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Print every epoch for Kaggle (can see progress)
    print(f"{epoch:<8} {train_loss:<12.4f} {val_loss:<12.4f} {val_acc:<12.4f} {status}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\n⏸️  Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
        break

# Training complete
elapsed_time = time.time() - start_time

print("\n" + "="*80)
print("🎉 TRAINING COMPLETE!")
print("="*80)
print(f"Best Validation Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")
print(f"Total Training Time: {elapsed_time/60:.2f} minutes")
print(f"Model saved to: classifier.pth")
print("="*80)

## 1️⃣2️⃣ Visualize Training Progress (Optional)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot losses
epochs = range(1, len(train_losses) + 1)
ax1.plot(epochs, train_losses, 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs, val_losses, 'r-', label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot accuracy
ax2.plot(epochs, val_accuracies, 'g-', linewidth=2)
ax2.axhline(y=best_val_acc, color='r', linestyle='--', label=f'Best: {best_val_acc:.4f}')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Validation Accuracy', fontsize=12)
ax2.set_title('Validation Accuracy Over Time', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Training Summary:")
print(f"   Best epoch: {best_epoch}")
print(f"   Best validation accuracy: {best_val_acc:.4f}")
print(f"   Final train loss: {train_losses[-1]:.4f}")
print(f"   Final val loss: {val_losses[-1]:.4f}")

## 1️⃣3️⃣ Download Your Trained Model 📥

Your trained model `classifier.pth` is ready! Here's how to download it from Kaggle:

### **Method 1: Direct Download (Recommended)**
1. Look at the right sidebar → "Output" section
2. Find `classifier.pth` file
3. Click the download button

### **Method 2: Verify File Exists**
Run the cell below to verify the file exists and see its size.

In [ ]:
import os

# Check if model file exists
model_file = 'classifier.pth'

if os.path.exists(model_file):
    file_size = os.path.getsize(model_file) / (1024 * 1024)  # Convert to MB
    print("="*80)
    print("✅ MODEL FILE READY FOR DOWNLOAD!")
    print("="*80)
    print(f"📄 Filename: {model_file}")
    print(f"📊 Size: {file_size:.2f} MB")
    print(f"🎯 Validation Accuracy: {best_val_acc:.4f}")
    print("\n📥 HOW TO DOWNLOAD:")
    print("   1. Look at the right sidebar → 'Output' section")
    print("   2. Find 'classifier.pth' in the file list")
    print("   3. Click the download button")
    print("\n💡 NEXT STEPS:")
    print("   1. Download classifier.pth to your local machine")
    print("   2. Place it in the same directory as app.py")
    print("   3. Run: streamlit run app.py")
    print("   4. Your application is now ready! 🎉")
    print("="*80)
else:
    print("❌ Model file not found! Make sure training completed successfully.")

---

## 🎉 Congratulations! You've Successfully Trained Your GNN Model!

### **What You Accomplished:**
✅ Built a heterogeneous knowledge graph with Papers, Authors, and Concepts  
✅ Created semantic embeddings for 2,000+ research papers  
✅ Trained a Graph Neural Network using GPU acceleration  
✅ Achieved **{best_val_acc:.1%}** validation accuracy  
✅ Generated a production-ready model file (`classifier.pth`)  

### **Project Completion:**
Your **Aura: Academic Research Assistant** is now complete with:
1. ✅ **Semantic Plagiarism Detector** - Find similar research papers
2. ✅ **Research Discovery Engine** - Identify experts by topic

### **Model Performance Interpretation:**
- **85-90% accuracy**: Excellent! Model understands paper-concept relationships well
- **80-85% accuracy**: Good! Model is production-ready
- **75-80% accuracy**: Acceptable! Consider more training epochs or data
- **<75% accuracy**: May need hyperparameter tuning

### **Important Files:**
- `classifier.pth` - Your trained GNN model (download this!)
- `nodes.csv` - Keep this for the application
- `relationships.csv` - Keep this for the application

### **Resources:**
- 📖 [PyTorch Geometric Docs](https://pytorch-geometric.readthedocs.io/)
- 📖 [SentenceTransformers Docs](https://www.sbert.net/)
- 🚀 [Streamlit Docs](https://docs.streamlit.io/)

---

**Made with ❤️ using Kaggle's Free GPU! 🚀**